# H₂ simulation test bench

Cleaned and organized for reproducible execution from the repository root.


In [ ]:
# ===== 第一步：使用 OpenFermion 构造 H2 Toy 模型的费米子哈密顿量 =====
from openfermion.ops import FermionOperator
from openfermion.transforms import bravyi_kitaev
from openfermion.utils import hermitian_conjugated
from qutip import *
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.animation import FuncAnimation
from mpl_toolkits.mplot3d import Axes3D

In [ ]:
# 构造简化 H2 分子的 toy Hamiltonian
H_f = FermionOperator('', -0.8126)
H_f += FermionOperator('0^ 0', -0.0458)
H_f += FermionOperator('1^ 1', -0.0458)
H_f += FermionOperator('2^ 2', 0.1743)
H_f += FermionOperator('3^ 3', 0.1743)
H_f += FermionOperator('0^ 1^ 1 0', 0.1220)
H_f += FermionOperator('2^ 3^ 3 2', 0.1676)
H_f += FermionOperator('0^ 2^ 2 0', 0.1686)
H_f += FermionOperator('1^ 3^ 3 1', 0.1686)
H_f += FermionOperator('0^ 3^ 3 0', 0.1202)
H_f += FermionOperator('1^ 2^ 2 1', 0.1202)
H_f += FermionOperator('0^ 2', -0.0450)
H_f += FermionOperator('1^ 3', -0.0450)
H_f += hermitian_conjugated(H_f)

In [ ]:
# ===== 第二步：使用 BK 变换得到泡利量子比特哈密顿量 =====
from openfermion.transforms import bravyi_kitaev

H_q = bravyi_kitaev(H_f)

# 将 QubitOperator 转换为 QuTiP 的 Qobj
n_qubits = 4
def qubit_operator_to_qutip(qubit_op, n_qubits):
    pauli_map = {'X': sigmax(), 'Y': sigmay(), 'Z': sigmaz()}
    H = 0
    for term, coeff in qubit_op.terms.items():
        ops = [qeye(2)] * n_qubits
        for idx, op in term:
            ops[idx] = pauli_map[op]
        H += coeff * tensor(ops)
    return H

H_qubit = qubit_operator_to_qutip(H_q, n_qubits)

In [ ]:
# ===== 第三步：构造两个玻色子振动模 =====
n_vib = 5
w1, w2 = 1.0, 1.2
J = 0.1
a1 = destroy(n_vib)
a2 = destroy(n_vib)
H_vib = (
    w1 * tensor(a1.dag()*a1, qeye(n_vib)) +
    w2 * tensor(qeye(n_vib), a2.dag()*a2) +
    J * (tensor(a1.dag(), a2) + tensor(a1, a2.dag()))
)


In [ ]:
# ===== 第四步：构造电子-振动模耦合项 =====
sigma_z = tensor(sigmaz(), qeye(2), qeye(2), qeye(2))
H_couple = tensor(sigma_z, tensor(a1 + a1.dag(), qeye(n_vib))) * 0.2

In [ ]:
# ===== 第五步：构造总哈密顿量 H_tot =====
I_q = tensor([qeye(2) for _ in range(n_qubits)])
I_b = qeye(n_vib)
I_bb = tensor(I_b, I_b)

H_tot = tensor(H_qubit, I_bb) + tensor(I_q, H_vib) + H_couple

In [ ]:
# ===== 第六步：初始化系统初态 psi0 =====
psi_qubit = tensor([basis(2, 0) for _ in range(n_qubits)])
psi_boson = tensor(basis(n_vib, 1), basis(n_vib, 0))
psi0 = tensor(psi_qubit, psi_boson)

In [ ]:
# ===== 第七步：设置时间与观测量，运行 mesolve 模拟 =====
tlist = np.linspace(0, 50, 200)
n1 = tensor(I_q, tensor(a1.dag() * a1, qeye(n_vib)))
n2 = tensor(I_q, tensor(qeye(n_vib), a2.dag() * a2))
collapse_ops = []

result = mesolve(H_tot, psi0, tlist, collapse_ops, [n1, n2])

In [ ]:
# ===== 第八步：绘图展示能量转移轨迹 =====
plt.plot(tlist, result.expect[0], label="⟨a1†a1⟩")
plt.plot(tlist, result.expect[1], label="⟨a2†a2⟩")
plt.xlabel("Time")
plt.ylabel("Occupation")
plt.title("H2 vibronic mode energy transfer")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# ===== 加入玻色子耗散项 =====
gamma1 = 0.02  # 模1耗散速率
gamma2 = 0.02  # 模2耗散速率
c_ops = [tensor(I_q, tensor(a1, I_b)) * np.sqrt(gamma1),
         tensor(I_q, tensor(I_b, a2)) * np.sqrt(gamma2)]

result = mesolve(H_tot, psi0, tlist, c_ops, [n1, n2])

# ===== 第八步：绘图展示能量转移轨迹 =====
plt.plot(tlist, result.expect[0], label="⟨a1†a1⟩")
plt.plot(tlist, result.expect[1], label="⟨a2†a2⟩")
plt.xlabel("Time")
plt.ylabel("Occupation")
plt.title("H₂ Vibronic Energy Transfer with Dissipation")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# ===== 加入玻色子耗散项（含非零温度热激发） =====
gamma1, gamma2 = 0.02, 0.02
n_th = 1.5  # 热浴激发数
c_ops = [
    np.sqrt(gamma1 * (n_th + 1)) * tensor(I_q, tensor(a1, I_b)),
    np.sqrt(gamma1 * n_th)       * tensor(I_q, tensor(a1.dag(), I_b)),
    np.sqrt(gamma2 * (n_th + 1)) * tensor(I_q, tensor(I_b, a2)),
    np.sqrt(gamma2 * n_th)       * tensor(I_q, tensor(I_b, a2.dag())),
]

result = mesolve(H_tot, psi0, tlist, c_ops, [n1, n2])

# ===== 第八步：绘图展示能量转移轨迹 =====
plt.plot(tlist, result.expect[0], label="⟨a1†a1⟩")
plt.plot(tlist, result.expect[1], label="⟨a2†a2⟩")
plt.xlabel("Time")
plt.ylabel("Occupation")
plt.title("H₂ Vibronic Energy Transfer with Thermal Dissipation")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
# ===== 第七步：设置时间与观测量，运行多个热耗散条件下的模拟（包含电子耗散） =====
tlist = np.linspace(0, 50, 200)
n1 = tensor(I_q, tensor(a1.dag() * a1, qeye(n_vib)))
n2 = tensor(I_q, tensor(qeye(n_vib), a2.dag() * a2))

results = {}
for n_th in [0.0, 0.5, 1.5, 3.0, 6, 12]:
    gamma1, gamma2 = 0.02, 0.02
    c_ops = [
        np.sqrt(gamma1 * (n_th + 1)) * tensor(I_q, tensor(a1, I_b)),
        np.sqrt(gamma1 * n_th)       * tensor(I_q, tensor(a1.dag(), I_b)),
        np.sqrt(gamma2 * (n_th + 1)) * tensor(I_q, tensor(I_b, a2)),
        np.sqrt(gamma2 * n_th)       * tensor(I_q, tensor(I_b, a2.dag())),
        # ===== 添加电子系统退相干（示例：第一个qubit T1） =====
        np.sqrt(0.01) * tensor(sigmax(), qeye(2), qeye(2), qeye(2), I_bb),
    ]
    res = mesolve(H_tot, psi0, tlist, c_ops, [n1, n2])
    results[n_th] = res.expect

# ===== 第八步：绘图展示不同热激发下的转移轨迹（a1, a2） =====
colors = ['tab:blue', 'tab:green', 'tab:orange', 'tab:red', 'tab:purple', 'tab:pink']
plt.figure(figsize=(10, 6))
for idx, (n_th, expect) in enumerate(results.items()):
    plt.plot(tlist, expect[0], linestyle='-', color=colors[idx], label=f'n_th={n_th} ⟨a1†a1⟩')
    plt.plot(tlist, expect[1], linestyle='--', color=colors[idx], label=f'n_th={n_th} ⟨a2†a2⟩')

plt.xlabel("Time")
plt.ylabel("Occupation")
plt.title("H₂ Vibronic Energy Transfer with Thermal and Electronic Dissipation")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
summary_data = []
for n_th, expect in results.items():
    avg_a1 = np.mean(expect[0])
    avg_a2 = np.mean(expect[1])
    summary_data.append({'n_th': n_th, 'avg_a1_occupation': avg_a1, 'avg_a2_occupation': avg_a2})

df = pd.DataFrame(summary_data)
df.to_csv("thermal_summary.csv", index=False)
print("Saved to thermal_summary.csv")